## Resumen ejecutivo

Un equipo de operaciones logísticas planea un ensayo de campo aleatorizado que compara tres estrategias de enrutamiento de conductores (línea base estática, reruteo dinámico, optimizado por IA) en tres regiones de depósito, con el retraso medio de entrega (minutos) como respuesta. PROC GLMPOWER toma un conjunto de datos *ejemplar* de medias de celda conjeturadas y resuelve el número total de conductores necesarios para detectar los efectos de estrategia con un 80% y un 90% de potencia, y luego traza cómo se erosiona la potencia alcanzable a medida que crece la variabilidad de ruta a ruta.

# Dimensionamiento de un ensayo de campo de enrutamiento de conductores con PROC GLMPOWER

## Resumen ejecutivo

Un equipo de operaciones logísticas está por lanzar un ensayo de campo aleatorizado que compara tres estrategias de enrutamiento de conductores -- una línea base **Estática**, un sistema de reruteo **Dinámico**, y un planificador **Optimizado por IA** -- desplegadas en tres regiones de depósito (Noreste, Medio Oeste, Oeste). La respuesta es el **retraso medio de entrega en minutos**. Antes de comprometer capacidad de flota al ensayo, el equipo debe responder: *¿cuántos conductores necesitamos para detectar de forma confiable la mejora operativa que esperamos?*

Este cuaderno usa **PROC GLMPOWER** para realizar un análisis prospectivo de potencia y tamaño de muestra para el modelo lineal general detrás del ensayo. Partiendo de un conjunto de datos *ejemplar* de medias de celda conjeturadas, (1) resuelve la inscripción total necesaria para alcanzar un 80% y un 90% de potencia para los efectos ómnibus de estrategia y región, (2) traza cómo se degrada la potencia alcanzable a medida que aumenta la variabilidad de ruta a ruta, y (3) produce una curva de potencia para respaldar la decisión de inscripción.

> **Conclusión clave:** Con una desviación estándar del error plausible de 8 minutos, aproximadamente **63 conductores** logran un 80% de potencia y **83 conductores** logran un 90% de potencia para detectar los efectos de la estrategia de enrutamiento -- pero si la variabilidad del retraso se acerca a 10 minutos, incluso 90 conductores inscritos se quedan por debajo del 70% de potencia, lo que subraya el valor de rutas ajustadas y bien instrumentadas.

---
## 1. Construcción del diseño ejemplar

El primer paso codifica el diseño del ensayo y el retraso medio *conjeturado* por el equipo para cada combinación de estrategia de enrutamiento x región de depósito. Fijamos una semilla aleatoria y agregamos una fluctuación insignificante para que las medias se vean realistas mientras se preserva la estructura balanceada 3x3. El peso `cell_n` de 1 en cada celda le indica a GLMPOWER que el diseño está balanceado.

In [1]:
/* ================================================================
   Generar el conjunto de datos ejemplar de retrasos medios conjeturados.
   Una fila por celda de diseño estrategia-de-enrutamiento x región-de-depósito.
   ================================================================ */
DATOS routing_trial;
   LLAMAR streaminit(20260531);
   LONGITUD routing_strategy $8 depot_region $9;
   ARREGLO strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   ARREGLO region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   ARREGLO smean[3]     _temporary_ (18.0 14.5 11.0);   /* medias de estrategia conjeturadas */
   ARREGLO radj[3]      _temporary_ (1.5  0.0 -1.0);    /* ajustes regionales (min)  */
   HACER i = 1 HASTA 3;
      HACER j = 1 HASTA 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         SALIDA;
      END;
   END;
   ELIMINAR i j;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=routing_trial label noobs;
   VAR routing_strategy depot_region mean_delay_min cell_n;
   ETIQUETA routing_strategy="Estrategia de enrutamiento" depot_region="Región del depósito"
         mean_delay_min="Retraso medio de entrega (min)" cell_n="Peso de celda";
   TÍTULO "Medias de celda ejemplares: retraso de entrega conjeturado (minutos)";
EJECUTAR;

                          Medias de celda ejemplares: retraso de entrega conjeturado (minutos)                          

Estrategia de enrutamiento    Región del depósito  Retraso medio de entrega (min)  Peso de celda
Static                      Northeast                                19.687507651              1
Static                      Midwest                                 17.9871067398              1
Static                      West                                    16.8240210086              1
Dynamic                     Northeast                               15.9537535365              1
Dynamic                     Midwest                                 14.4415169858              1
Dynamic                     West                                    12.8636389493              1
AIOpt                       Northeast                               12.6143811724              1
AIOpt                       Midwest                                  10.893885771              1
AIOpt


NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Tamaño de muestra para el diseño ómnibus

Con el diseño en mano, le pedimos a GLMPOWER que **resuelva el tamaño de muestra total** (`NTOTAL = .`) con un 80% y un 90% de potencia. La instrucción `MODEL` especifica un diseño de dos vías con interacción (`routing_strategy*depot_region`), `WEIGHT cell_n` declara los pesos de perfil balanceados, y `STDDEV = 8` es el error cuadrático medio asumido del retraso de entrega. `NFRACTIONAL` permite que el procedimiento reporte conteos fraccionarios exactos antes de redondear hacia arriba.

También pre-registramos tres comparaciones `CONTRAST` planeadas -- IA-Optimizado vs. Estática, Dinámica vs. Estática, y cualquier reruteo vs. Estática -- que documentan las hipótesis operativamente significativas que el ensayo está construido para probar.

In [2]:
/* ================================================================
   Resolver el total de conductores necesarios para alcanzar 80% y 90%
   de potencia para los efectos de estrategia de enrutamiento y región.
   ================================================================ */
PROCEDIMIENTO glmpower DATOS=routing_trial;
   CLASE routing_strategy depot_region;
   MODELO mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   PESO cell_n;
   ETIQUETA routing_strategy="Estrategia de enrutamiento" depot_region="Región del depósito"
         mean_delay_min="Retraso medio de entrega (min)";
   CONTRAST "IA-Optimizado vs. Estática"     routing_strategy -1  0  1;
   CONTRAST "Dinámica vs. Estática"          routing_strategy -1  1  0;
   CONTRAST "Cualquier reruteo vs. Estática" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TÍTULO "Tamaño de muestra para detectar efectos de la estrategia de enrutamiento sobre el retraso";
EJECUTAR;

                          Medias de celda ejemplares: retraso de entrega conjeturado (minutos)                          

The GLMPOWER Procedure


                      Fixed Scenario Elements                       

Item                Value                                           
------------------  ------------------------------------------------
Dependent Variable  Retraso medio de entrega (min)                  
Source              Estrategia de enrutamiento                      
Source              Región del depósito                             
Source              Estrategia de enrutamiento*Región del depósito  

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Sensibilidad de la potencia a la variabilidad y al tamaño del ensayo

La respuesta del tamaño de muestra depende de la desviación estándar del error asumida, que rara vez se conoce con precisión de antemano. Aquí invertimos la pregunta: **fijamos** varios totales de inscripción candidatos (`NTOTAL = 45 90 180`) y **resolvemos la potencia lograda** (`POWER = .`) a través de una grilla de desviaciones estándar de retraso plausibles (6, 8 y 10 minutos). La tabla resultante es un mapa de sensibilidad -- muestra qué tan robusto es cada plan de inscripción si la variabilidad de ruta real resulta ser mayor de lo esperado.

In [3]:
/* ================================================================
   Grilla de sensibilidad: potencia lograda a través de tamaños de
   ensayo candidatos y desviaciones estándar de retraso plausibles.
   ================================================================ */
PROCEDIMIENTO glmpower DATOS=routing_trial;
   CLASE routing_strategy depot_region;
   MODELO mean_delay_min = routing_strategy depot_region;
   PESO cell_n;
   ETIQUETA routing_strategy="Estrategia de enrutamiento" depot_region="Región del depósito"
         mean_delay_min="Retraso medio de entrega (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TÍTULO "Potencia lograda según escenarios de variabilidad y tamaño del ensayo";
EJECUTAR;

                          Medias de celda ejemplares: retraso de entrega conjeturado (minutos)                          

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  Retraso medio de entrega (min)
Source              Estrategia de enrutamiento    
Source              Región del depósito           

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Curva de potencia para la decisión de inscripción

Finalmente, trazamos una **curva de potencia** -- potencia lograda a medida que la inscripción total crece de 30 a 270 conductores en pasos de 30 -- para el modelo de estrategia más región a la desviación estándar esperada de 8 minutos. Resolver `POWER = .` a través de esa grilla de inscripción produce la curva como una serie tabulada de `N Total` vs. `Power`, para que podamos leer en qué inscripción se cruza cada uno de los objetivos convencionales del 80% y 90%.

In [4]:
/* ================================================================
   Curva de potencia: potencia lograda vs. total de conductores
   inscritos, barrido de 30 a 270 en pasos de 30 a la desviación
   estándar esperada de 8 min.
   ================================================================ */
PROCEDIMIENTO glmpower DATOS=routing_trial;
   CLASE routing_strategy depot_region;
   MODELO mean_delay_min = routing_strategy depot_region;
   PESO cell_n;
   ETIQUETA routing_strategy="Estrategia de enrutamiento" depot_region="Región del depósito"
         mean_delay_min="Retraso medio de entrega (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TÍTULO "Curva de potencia: potencia lograda vs. total de conductores inscritos";
EJECUTAR;

                          Medias de celda ejemplares: retraso de entrega conjeturado (minutos)                          

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  Retraso medio de entrega (min)
Source              Estrategia de enrutamiento    
Source              Región del depósito           

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretación y recomendación

El análisis le da al equipo de operaciones un plan de inscripción defendible:

- **Dimensionamiento de referencia (Sección 2).** Asumiendo una desviación estándar residual de 8 minutos, el modelo completo de dos vías (estrategia, región y su interacción) alcanza un **80% de potencia con 63 conductores** y un **90% de potencia con 83 conductores**. Redondeando hacia arriba por deserción, un objetivo de inscripción cercano a **90 conductores** supera cómodamente el umbral del 90%.

- **La sensibilidad importa (Sección 3).** La potencia es muy sensible a la variabilidad del retraso. Con 90 conductores, la potencia lograda cae de ~99% (DE=6) a ~87% (DE=8) a ~68% (DE=10). Un piloto de 45 conductores solo es adecuado si la variabilidad es baja (~81% de potencia con DE=6) y queda gravemente subdimensionado (~37%) si la DE llega a 10. La implicación práctica: invertir en rutas consistentes y bien instrumentadas para mantener baja la variabilidad es tan valioso como agregar conductores.

- **La curva de potencia (Sección 4).** Trazada para el modelo de estrategia más región (sin término de interacción, la óptica usada para el barrido de sensibilidad), la potencia lograda sube de 0.37 con 30 conductores a 0.69 con 60, 0.87 con 90 y 0.95 con 120, aplanándose por encima de 0.99 con 180. Leyendo la curva contra los objetivos, el 80% de potencia llega cerca de 77 conductores y el 90% cerca de 99 -- un poco más alto que el dimensionamiento del modelo completo en la Sección 2 porque quitar el término de interacción distribuye el mismo efecto entre menos grados de libertad del modelo.

**Recomendación:** Inscribir aproximadamente **90 conductores** (30 por estrategia de enrutamiento, balanceados entre las tres regiones de depósito). Esto supera el 90% de potencia para el modelo completo bajo la desviación estándar esperada de 8 minutos, mantiene ~87% de potencia incluso en la curva del modelo reducido más conservadora, y mantiene el ensayo lo bastante pequeño como para ejecutarlo dentro de un solo trimestre operativo.

*Nota:* GLMPOWER analiza el *diseño* conjeturado, no los resultados observados -- así que la credibilidad de estas cifras depende de las medias y la desviación estándar conjeturadas. Los equipos deberían revisar el dimensionamiento una vez que un breve piloto arroje una estimación empírica de la variabilidad del retraso de entrega.